In [ ]:
# Setup

# pip.main(['install', 'splink'])
# pip.main(['install', 'pyspark'])
# pip.main(['install', 'duckdb'])
# pip.main(['install', 'pyarrow'])
# pip.main(['install', 'pandas'])
# pip.main(['install', 'matplotlib'])

In [1]:
# package imports

from itertools import count
import pip
from requests import head
import splink
import pyspark 
import pandas as pd
import re 
import pyarrow as pa


# SPINK setup
import splink.comparison_library as cl
import splink.comparison_level_library as cll
from splink.exploratory import profile_columns
from splink.comparison_library import CustomComparison
import duckdb, os, tempfile
import sys
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
from splink.exploratory import completeness_chart
import csv



In [2]:
# functions


def cleanse_names(series: pd.Series) -> pd.Series:
    """
    Clean text columns similar to your Spark UDF logic.
    """
    # lowercase
    cleaned = series.str.lower()

    # remove special characters (keep only letters, digits, space)
    cleaned = cleaned.str.replace(r"[^a-z0-9 ]", "", regex=True)

    # normalize whitespace
    cleaned = cleaned.str.strip().str.replace(r"\s+", " ", regex=True)

    return cleaned


In [3]:
# DuckDB setup

# Set up DuckDB in memory
# In theory we can set this to a path on the local drive, but it will be slower
con = duckdb.connect(":memory:")

# Set up temporary dir for disk spilling.
spill_dir = tempfile.mkdtemp(prefix="duckdb_spill_")
con.execute("SET memory_limit = '100GB';")  # synonyms: max_memory / memory_limit
con.execute(f"SET temp_directory = '{spill_dir}';")
con.execute("SET max_temp_directory_size = '200GB';")

# This gets used across various splink functions
db_api = DuckDBAPI(con)

In [4]:

# read gias data
gias = pd.read_csv('Data/gias_data2024-03-01_2024-09-01_28.csv')

C:\Users\spilling\AppData\Local\Temp\ipykernel_26352\2644862179.py:2: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  gias = pd.read_csv('Data/gias_data2024-03-01_2024-09-01_28.csv')


In [5]:
    # Ensure all columns are strings for Splink
for col in gias.columns:
        gias[col] = gias[col].astype(str)

In [6]:
#clean nulls
for col_name in gias.columns:
    print(f"Cleaning column: {col_name}")
    # Trim whitespace
    gias[col_name] = gias[col_name].astype(str).str.strip()
    # Replace null-like values with pd.NA
    gias[col_name] = gias[col_name].replace(
        to_replace=[r"^\s*$", r"^NA$", r"^NA NA$", r"^na$", r"^NaN$", r"^nan$", r"^N/A$", r"^n/a$", r"^null$", r"^NULL$", r"^<NA>$",r"^none$",r"^None$"],
        value=pd.NA,
        regex=True
    )

Cleaning column: Unnamed: 0
Cleaning column: urn
Cleaning column: establishment_name
Cleaning column: previous_la_code
Cleaning column: previous_establishment_number
Cleaning column: northing
Cleaning column: easting
Cleaning column: postcode
Cleaning column: ukprn
Cleaning column: establishment_type_group_name
Cleaning column: phase_of_education_name
Cleaning column: establishment_status_name
Cleaning column: trusts_name
Cleaning column: official_sixth_form_name
Cleaning column: administrative_ward_name
Cleaning column: msoa_name
Cleaning column: lsoa_name
Cleaning column: uprn
Cleaning column: school_capacity
Cleaning column: gender_name
Cleaning column: statutory_low_age
Cleaning column: statutory_high_age
Cleaning column: open_date
Cleaning column: close_date
Cleaning column: gias_date
Cleaning column: laestab
Cleaning column: previous_laestab
Cleaning column: heads_name


In [7]:
sum(gias["heads_name"] == "na")

0

In [ ]:

cols_to_clean = [
    "heads_name",
    "establishment_name",
    "previous_establishment_number",
    "trusts_name",
]

for c in cols_to_clean:
    gias[c] = cleanse_names(gias[c].astype(str))

# drop unnamed index column if it exists

gias = gias.drop(columns=['Unnamed: 0'])

# Drop duplicates ignoring 'gias_date' column

subset = gias.columns.difference(['gias_date'])

gias = gias.drop_duplicates(subset=subset)

# easting and northings 0.0 to null

gias['easting'] = gias['easting'].replace('0.0', pd.NA)
gias['northing'] = gias['northing'].replace('0.0', pd.NA)


gias["unique_id"] = range(1, len(gias) + 1)



In [ ]:
#clean nulls after function
for col_name in gias.columns:
    print(f"Cleaning column: {col_name}")
    # Trim whitespace
    gias[col_name] = gias[col_name].astype(str).str.strip()
    # Replace null-like values with pd.NA
    gias[col_name] = gias[col_name].replace(
        to_replace=[r"^\s*$", r"^NA$", r"^NA NA$", r"^na$", r"^NaN$", r"^nan$", r"^N/A$", r"^n/a$", r"^null$", r"^NULL$", r"^<NA>$",r"^none$",r"^None$"],
        value=pd.NA,
        regex=True
    )

Cleaning column: urn
Cleaning column: establishment_name
Cleaning column: previous_la_code
Cleaning column: previous_establishment_number
Cleaning column: northing
Cleaning column: easting
Cleaning column: postcode
Cleaning column: ukprn
Cleaning column: establishment_type_group_name
Cleaning column: phase_of_education_name
Cleaning column: establishment_status_name
Cleaning column: trusts_name
Cleaning column: official_sixth_form_name
Cleaning column: administrative_ward_name
Cleaning column: msoa_name
Cleaning column: lsoa_name
Cleaning column: uprn
Cleaning column: school_capacity
Cleaning column: gender_name
Cleaning column: statutory_low_age
Cleaning column: statutory_high_age
Cleaning column: open_date
Cleaning column: close_date
Cleaning column: gias_date
Cleaning column: laestab
Cleaning column: previous_laestab
Cleaning column: heads_name
Cleaning column: unique_id


In [14]:
print(gias.columns)

print(len(gias))

Index(['urn', 'establishment_name', 'previous_la_code',
       'previous_establishment_number', 'northing', 'easting', 'postcode',
       'ukprn', 'establishment_type_group_name', 'phase_of_education_name',
       'establishment_status_name', 'trusts_name', 'official_sixth_form_name',
       'administrative_ward_name', 'msoa_name', 'lsoa_name', 'uprn',
       'school_capacity', 'gender_name', 'statutory_low_age',
       'statutory_high_age', 'open_date', 'close_date', 'gias_date', 'laestab',
       'previous_laestab', 'heads_name', 'unique_id'],
      dtype='object')
56172


In [59]:
filtered_gias = gias[gias["urn"] == "132454"]
display(filtered_gias["trusts_name"])


32281    <NA>
Name: trusts_name, dtype: object

In [15]:

completeness_chart(
    gias,
    db_api=db_api)

alt.LayerChart(...)

In [20]:
profile_columns(gias, db_api=db_api, column_expressions=["urn"])

alt.VConcatChart(...)

In [ ]:
profile_columns(gias, db_api=db_api, column_expressions=["easting"])

alt.VConcatChart(...)

In [60]:
profile_columns(gias, db_api=db_api, column_expressions=["trusts_name"])

alt.VConcatChart(...)

In [18]:
profile_columns(gias, db_api=db_api, column_expressions=["ukprn"])

alt.VConcatChart(...)

In [ ]:
profile_columns(gias, db_api=db_api, column_expressions=["heads_name"])

alt.VConcatChart(...)

In [21]:
blocking_rules_dedupe = [
  block_on("urn"),
  block_on("establishment_name"),
  block_on("laestab"),
#   block_on("ukprn"),
  block_on("phase_of_education_name", "trusts_name"),
   block_on("northing", "easting"),
   block_on("postcode"),
   block_on("heads_name"),
]

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
  table_or_tables=gias,
  blocking_rules=blocking_rules_dedupe,
  db_api=db_api,
  link_type="dedupe_only",
)


alt.Chart(...)

In [22]:
# custom comparison for full_name

headteacher_name_comparison = CustomComparison(
    output_column_name="heads_name",
    comparison_levels=[
        cll.NullLevel("heads_name"),
        cll.ExactMatchLevel("heads_name").configure(tf_adjustment_column="heads_name"),
        cll.JaroWinklerLevel("heads_name", 0.9).configure(tf_adjustment_column="heads_name"),
        cll.ElseLevel(),
    ],
)

northing_easting_comparison = CustomComparison(
     output_column_name="northing_easting_distance",
     comparison_levels=[
         cll.And(cll.NullLevel("easting"),cll.NullLevel("northing")),  # level 0: nulls
         cll.And(cll.ExactMatchLevel("easting"),cll.ExactMatchLevel("northing")),
         cll.CustomLevel(
             """ (easting_l IS NOT NULL AND northing_l IS NOT NULL AND
                  easting_r IS NOT NULL AND northing_r IS NOT NULL AND
                  SQRT((CAST(easting_l as DOUBLE) - CAST(easting_r as DOUBLE))*(CAST(easting_l as DOUBLE) - CAST(easting_r as DOUBLE)) +
                   (CAST(northing_l as DOUBLE) - CAST(northing_r as DOUBLE))*(CAST(northing_l as DOUBLE) - CAST(northing_r as DOUBLE))) < 350)
             """
         ),
         cll.ElseLevel()  # everything else
     ]
 )

In [64]:


settings = SettingsCreator(
    link_type="dedupe_only",
    unique_id_column_name="unique_id",
    blocking_rules_to_generate_predictions= blocking_rules_dedupe,
    comparisons=[
        cl.ExactMatch("urn"),
        cl.JaroWinklerAtThresholds("laestab"),
        cl.PostcodeComparison("postcode",km_thresholds=[1, 10, 100]),
        cl.NameComparison("establishment_name"),
        headteacher_name_comparison,
        northing_easting_comparison
    ],
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    gias,
    settings,
    db_api=db_api,
    validate_settings=True
)

In [65]:

linker.training.estimate_probability_two_random_records_match(
    [
        block_on("urn")    ],
    recall=0.98,
)

Probability two random records match is estimated to be  3.86e-06.
This means that amongst all possible pairwise record comparisons, one in 259,233.12 are expected to match.  With 1,577,618,706 total possible comparisons, we expect a total of around 6,085.71 matching pairs


In [66]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e7)

----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - urn (no m values are trained).
    - laestab (no m values are trained).
    - postcode (no m values are trained).
    - establishment_name (no m values are trained).
    - heads_name (no m values are trained).
    - northing_easting_distance (no m values are trained).


In [67]:
training_session_names = (
    linker.training.estimate_parameters_using_expectation_maximisation(
        blocking_rule = block_on("urn")
    )
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."urn" = r."urn"

Parameter estimates will be made for the following comparison(s):
    - laestab
    - postcode
    - establishment_name
    - heads_name
    - northing_easting_distance

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - urn

Iteration 1: Largest change in params was 0.509 in probability_two_random_records_match
Iteration 2: Largest change in params was 3.98e-05 in probability_two_random_records_match

EM converged after 2 iterations

Your model is not yet fully trained. Missing estimates for:
    - urn (no m values are trained).


In [68]:
linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("establishment_name"),
)

linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("laestab"),
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."establishment_name" = r."establishment_name"

Parameter estimates will be made for the following comparison(s):
    - urn
    - laestab
    - postcode
    - heads_name
    - northing_easting_distance

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - establishment_name

Iteration 1: Largest change in params was -0.54 in the m_probability of urn, level `Exact match on urn`
Iteration 2: Largest change in params was -0.00744 in the m_probability of laestab, level `Exact match on laestab`
Iteration 3: Largest change in params was -0.000282 in the m_probability of laestab, level `Exact match on laestab`
Iteration 4: Largest change in params was -1.52e-05 in the m_probability of laestab, level `Exact match on laestab`

EM converged after 4 iterations

Your model is fully trained. All comparisons have at least one estimate

<EMTrainingSession, blocking on l."laestab" = r."laestab", deactivating comparisons laestab>

In [69]:
linker.visualisations.parameter_estimate_comparisons_chart()

alt.Chart(...)

In [70]:
linker.visualisations.match_weights_chart()

alt.VConcatChart(...)

In [71]:
linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

In [72]:
df_predict = linker.inference.predict()

df_e = df_predict.as_pandas_dataframe()

df_e = df_e.sort_values(by="match_probability", ascending=False)

display(df_e)


Blocking time: 0.17 seconds
Predict time: 1.12 seconds


,match_weight,match_probability,unique_id_l,unique_id_r,urn_l,urn_r,gamma_urn,bf_urn,laestab_l,laestab_r,...,easting_r,northing_l,northing_r,gamma_northing_easting_distance,bf_northing_easting_distance,trusts_name_l,trusts_name_r,phase_of_education_name_l,phase_of_education_name_r,match_key
57108,55.770304,1.000000e+00,10081,42640,110087,143803,0,0.599079,8714800,8714800,...,498170.0,180992.0,180992.0,2,47706.698249,None,st thomas catholic academies trust,Secondary,Secondary,1
128619,55.656076,1.000000e+00,47698,54945,149782,149782,1,105617.780071,9414002,9414002,...,477426.0,None,260718.0,-1,1.000000,northampton school for boys,the nsb trust,Secondary,Secondary,0
117009,73.782943,1.000000e+00,2985,51858,102986,102986,1,105617.780071,3192043,3192043,...,529920.0,163508.0,163508.0,2,47706.698249,sutton education trust,sutton education trust,Primary,Primary,0
144141,55.396293,1.000000e+00,51498,55895,148669,148669,1,105617.780071,8253361,8253361,...,484505.0,186980.0,186980.0,2,47706.698249,st thomas catholic academies trust,st thomas catholic academies trust,Primary,Primary,0
117010,70.975588,1.000000e+00,3427,51863,103430,103430,1,105617.780071,3303328,3303328,...,409596.0,280480.0,280480.0,2,47706.698249,None,None,Primary,Primary,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60016,-46.252600,1.192834e-14,41567,43245,142581,144498,0,0.599079,9372042,8602027,...,411391.0,254613.0,310475.0,0,0.008435,community academies trust,community academies trust,Primary,Primary,3
121387,-46.252600,1.192834e-14,40471,52318,141404,137910,0,0.599079,3732042,9313837,...,444918.0,388017.0,242396.0,0,0.008435,united learning trust,united learning trust,Primary,Primary,3
60037,-46.252600,1.192834e-14,42465,43280,143544,144536,0,0.599079,3722026,3713307,...,448493.0,400540.0,408197.0,0,0.008435,james montgomery academy trust,james montgomery academy trust,Primary,Primary,3
121376,-46.252600,1.192834e-14,39628,51766,140515,150691,0,0.599079,8664084,2094002,...,538311.0,186735.0,172429.0,0,0.008435,united learning trust,united learning trust,Secondary,Secondary,3


In [73]:
threshold = 0.6
edge_records = df_e[(df_e["match_probability"] > threshold - 0.2) & (df_e["match_probability"] < threshold + 0.2)]

display(edge_records)

,match_weight,match_probability,unique_id_l,unique_id_r,urn_l,urn_r,gamma_urn,bf_urn,laestab_l,laestab_r,...,easting_r,northing_l,northing_r,gamma_northing_easting_distance,bf_northing_easting_distance,trusts_name_l,trusts_name_r,phase_of_education_name_l,phase_of_education_name_r,match_key
98193,1.987609,0.798622,44930,48064,146422,150266,0,0.599079,9351103,9351117,...,583818.0,266011.0,265878.0,1,479.911348,None,raedwald trust,None,None,1
81927,1.884918,0.786931,40557,45886,141504,147610,0,0.599079,8667906,8667900,...,416877.0,185929.0,185931.0,1,479.911348,None,None,None,None,6
56646,1.832898,0.780823,39646,42054,140533,143109,0,0.599079,3052029,3052030,...,542188.0,171446.0,172762.0,0,0.008435,the spring partnership trust,None,Primary,Primary,5
91570,1.832898,0.780823,39927,45645,140819,147321,0,0.599079,9262059,9262148,...,637420.0,324857.0,325229.0,0,0.008435,broad horizons education trust,broad horizons education trust,Primary,Primary,3
145,1.832898,0.780823,401,426,100401,100426,0,0.599079,2062128,2062624,...,530952.0,183535.0,183516.0,0,0.008435,None,None,Primary,Primary,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79730,-0.568939,0.402669,15088,44219,115100,145602,0,0.599079,8813132,8813133,...,545847.0,208937.0,208635.0,0,0.008435,None,the diocese of chelmsford vine schools trust,Primary,Primary,5
38041,-0.568939,0.402669,15802,37643,115814,138430,0,0.599079,9167008,9167003,...,396612.0,202273.0,222959.0,0,0.008435,None,academies enterprise trust,None,None,5
46428,-0.568939,0.402669,20787,39249,120801,140118,0,0.599079,9262043,9262042,...,618703.0,310282.0,310314.0,0,0.008435,None,None,Primary,Primary,5
127157,-0.568939,0.402669,3283,54452,103286,151212,0,0.599079,3302236,3302223,...,406363.0,278942.0,279180.0,0,0.008435,None,None,Primary,Primary,5


In [74]:
records_to_plot = edge_records.head(200).to_dict(orient="records")

linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)

alt.LayerChart(...)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(
    df_e[(df_e['match_probability'] > 0.5) & (df_e['match_probability'] < 0.95)]['match_probability'],
    bins=50,
    edgecolor='black'
)
plt.xlabel('Match Probability')
plt.ylabel('Frequency')
plt.title('Histogram of Match Probability between 0.6 and 0.95')
plt.show()